# GATos Team - Step 1: Model-Ensemble

Reproduce Model-Ensemble training from the supplied training workbook using the same shared implementation as the final evaluation pipeline. This notebook generates the trained model artifacts required for prediction.
To evaluate the submitted system on a new or held-out Excel workbook using the supplied trained artifacts, use `GATos_FinalEvaluation_Colab.ipynb`.
Predict a workbook with retained models using the unchanged shared implementation.

Use **Google Colab Pro+ with Python 3.12.13 / A100 80 GB**. Verify the actual GPU allocation.
Place exported artifacts in `MyDrive/GATos_Team_Artifacts`.

Run all cells with `CURRENT_RUNTIME = "A"`. After PASS, select
**Runtime → Disconnect and delete runtime**, reconnect, set B, and rerun.
Repeat for C after B passes. Fresh VMs isolate incompatible dependencies.
Keep the same paths and run name. Outputs persist on Google Drive.
Never close or replace a runtime before its stage reports PASS.

Default: `pretrained`. Optional `fresh-training` runs the full procedure in
a separate workspace and takes much longer. Use pretrained for acceptance.

In [ ]:
# MOD001 - Mount Drive and configure paths.
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
CURRENT_RUNTIME = "A"
MODE = "pretrained"
ROOT = Path("/content/drive/MyDrive/GATos_Team")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/GATos_Team_Artifacts")
ARTIFACTS = ARTIFACT_ROOT / "Runs/fresh"
TRAIN_PATH = (ROOT if MODE == "fresh-training" else ARTIFACT_ROOT) / "Input/train.xlsx"
TEST_PATH = ROOT / "Input/test.xlsx"
RUN_NAME = "acceptance_01"
EVALUATION = ROOT / "Evaluations" / RUN_NAME
TRAINING_WORKSPACE = ROOT / "FreshTraining"

assert CURRENT_RUNTIME in {"A", "B", "C"}
assert MODE in {"pretrained", "fresh-training"}
assert (ROOT / "pyproject.toml").is_file(), f"Missing release: {ROOT}"
assert TRAIN_PATH.is_file(), f"Missing training workbook: {TRAIN_PATH}"
assert TEST_PATH.is_file(), f"Missing evaluation workbook: {TEST_PATH}"
print("Mode:", MODE)
print("Runtime:", CURRENT_RUNTIME)
print("Test input:", TEST_PATH)
print("Persistent evaluation:", EVALUATION)


`Input/test.xlsx` is public acceptance input. For unseen data, change
`TEST_PATH` and `RUN_NAME`. No public decisions are applied. Preserve the
training workbook bytes: pretrained uses the export, fresh-training the release.

In [ ]:
# MOD002 - Verify Python and GPU allocation.
import subprocess
import sys

if sys.version_info[:3] != (3, 12, 13):
    raise RuntimeError("Choose the recorded Python 3.12.13 Colab runtime before continuing.")
gpu = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits",
], text=True).strip()
name, memory = gpu.splitlines()[0].rsplit(",", 1)
if "A100" not in name or float(memory) < 75 * 1024:
    raise RuntimeError(f"Expected A100 80 GB, found: {gpu}")
print("GPU:", gpu)
print("Python:", sys.version)


In [ ]:
# MOD003 - Install this runtime's dependencies.
import subprocess
import sys

requirements = "requirements-probes.txt" if CURRENT_RUNTIME == "B" else "requirements-train.txt"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-r", str(ROOT / requirements),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(ROOT),
], check=True)
print("Installed environment:", requirements)

In [ ]:
# MOD004 - Stream commands and propagate failures.
import sys

sys.path.insert(0, str(ROOT / "src"))

def run_command(command):
    from gatos_release.launch import run_streamed
    run_streamed(command, root=ROOT)

def model_phase(phase):
    run_command([
        sys.executable, "-u", "-m", "gatos_release.model",
        "--root", ROOT,
        "--artifacts", ARTIFACTS,
        "--train", TRAIN_PATH,
        "--test", TEST_PATH,
        "--output", EVALUATION,
        phase,
    ])

In [ ]:
# MOD005 - Verify the packaged training runtime.
required_runtime_files = [
    ROOT / "config/pipeline.yaml",
    ROOT / "scripts/fresh_training.py",
    ROOT / "requirements-train.txt",
    ROOT / "requirements-probes.txt",
]
missing = [str(path) for path in required_runtime_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing training files: " + ", ".join(missing))

import gatos.cli
if CURRENT_RUNTIME == "B":
    import gatos.boundary_probes
else:
    import gatos.train
    import gatos.train_lora

print("TRAINING PACKAGE CHECK: PASS")

## Runtime A - prepare and QLoRA predictions

Reuse trained adapters for the supplied workbook. Small outputs persist
in the evaluation folder. No adapters are trained.

In [ ]:
# MOD006 - Prepare and run QLoRA inference.
if CURRENT_RUNTIME == "A" and MODE == "pretrained":
    model_phase("prepare")
    model_phase("qlora-test")
    print("Runtime A: PASS. Continue in a fresh Runtime B.")
else:
    print("Skipped: Runtime A pretrained stage.")

## Runtime B - fixed semantic-boundary probes

In a fresh VM, the frozen 72B model scores test posts only.
Retained training probes are reused.

In [ ]:
# MOD007 - Generate test boundary probes.
if CURRENT_RUNTIME == "B" and MODE == "pretrained":
    model_phase("probe-test")
    print("Runtime B: PASS. Continue in a fresh Runtime C.")
else:
    print("Skipped: Runtime B pretrained stage.")

## Runtime C - encoder inference and final assembly

Check checkpoints, run encoders, and assemble risk and evidence.
This is the heaviest inference phase. Do not rerun training.

In [ ]:
# MOD008 - Assemble and validate raw predictions.
if CURRENT_RUNTIME == "C" and MODE == "pretrained":
    model_phase("predict")
    model_phase("status")
else:
    print("Skipped: Runtime C pretrained stage.")

## Optional full training

Set `MODE = "fresh-training"`: A trains encoders and QLoRA, B scores
train/test probes, and C calibrates and predicts in `FreshTraining`.
No original checkpoints are imported. CUDA weights need not be identical.
See `docs/FRESH_TRAINING.md`.

In [ ]:
# MOD009 - Run optional isolated full training.
if MODE == "fresh-training":
    run_command([
        sys.executable, ROOT / "scripts/fresh_training.py",
        "--workspace", TRAINING_WORKSPACE,
        "--train", TRAIN_PATH,
        "--test", TEST_PATH,
        "--phase", CURRENT_RUNTIME,
    ])
else:
    print("Skipped: full training. Existing trained artifacts are retained.")


After Runtime C completes, open `GATos_Improver_Colab.ipynb` in `unseen-auto` mode. Use the same `TEST_PATH` and `RUN_NAME`, and set `MODEL_SOURCE = "fresh-training"` to process the raw predictions produced by the newly trained Model-Ensemble. The separate `public-replay` mode reproduces the historical reviewed public result and is not part of new-data evaluation.